In [0]:
USE CATALOG 'supermercadoetl_prod';
USE SCHEMA bronze;

In [0]:
-- MODELO PROPUESTO PARA SILVER (Pipeline diario con histórico)
-- 
-- ESTRATEGIA DE CARGA:
--   - producto_id: Identifica el producto (sin fecha) - permite MERGE
--   - fecha_extraccion: Dimensión temporal
--   - Clave compuesta: (producto_id, fecha_extraccion) = única
--   - MERGE diario: actualiza si existe, inserta si es nuevo
--
-- Columnas a mantener:
--   ✓ producto_id (surrogate key SIN fecha para histórico)
--   ✓ supermercado (dimensión de negocio)
--   ✓ nombre (atributo funcional)
--   ✓ descripcion (atributo funcional)
--   ✓ precio_normalizado (métrica de negocio limpia)
--   ✓ categoria (dimensión de negocio)
--   ✓ url (referencia funcional)
--   ✓ fecha_extraccion (dimensión temporal - parte de PK)
--   ✓ fecha_actualizacion (metadata para auditoría)
--
-- Columnas a ELIMINAR (técnicas de Bronze):
--   ✗ precio_raw (ya normalizado)
--   ✗ batch_id (control técnico de ingesta)
--   ✗ ingestion_timestamp (timestamp técnico)
--   ✗ load_date (metadata técnico redundante)
--   ✗ validation_status (usado solo en transformación)

SELECT
  -- Surrogate key SIN fecha (mismo producto = mismo ID)
  MD5(CONCAT(supermercado, '|', nombre)) AS producto_id,
  
  -- Dimensiones de negocio
  supermercado,
  categoria,
  
  -- Atributos funcionales del producto
  nombre,
  descripcion,
  url,
  
  -- Métrica de negocio (precio limpio y normalizado)
  CAST(
    REGEXP_REPLACE(
      REGEXP_REPLACE(TRIM(precio), '[^0-9,.-]', ''),
      ',',
      '.'
    ) AS DECIMAL(18,2)
  ) AS precio_normalizado,
  
  -- Dimensión temporal (parte de la clave compuesta)
  fecha_extraccion,
  
  -- Metadata de auditoria
  CURRENT_TIMESTAMP() AS fecha_actualizacion
  
FROM precios_scraping

-- Filtros de calidad para Silver
WHERE 
  -- Solo registros con precio válido
  TRY_CAST(
    REGEXP_REPLACE(
      REGEXP_REPLACE(TRIM(precio), '[^0-9,.-]', ''),
      ',',
      '.'
    ) AS DECIMAL(18,2)
  ) IS NOT NULL
  AND TRY_CAST(
    REGEXP_REPLACE(
      REGEXP_REPLACE(TRIM(precio), '[^0-9,.-]', ''),
      ',',
      '.'
    ) AS DECIMAL(18,2)
  ) > 0
  -- Campos críticos no nulos
  AND supermercado IS NOT NULL
  AND nombre IS NOT NULL
  AND categoria IS NOT NULL

LIMIT 10;

In [0]:
-- Validaciones adicionales para capa Silver
WITH datos_validos AS (
  SELECT
    supermercado,
    nombre,
    descripcion,
    precio AS precio_raw,
    TRY_CAST(
      REGEXP_REPLACE(
        REGEXP_REPLACE(TRIM(precio), '[^0-9,.-]', ''),
        ',',
        '.'
      ) AS DECIMAL(18,2)
    ) AS precio_num,
    categoria,
    url,
    fecha_extraccion,
    batch_id
  FROM precios_scraping
  WHERE TRY_CAST(
      REGEXP_REPLACE(
        REGEXP_REPLACE(TRIM(precio), '[^0-9,.-]', ''),
        ',',
        '.'
      ) AS DECIMAL(18,2)
    ) IS NOT NULL
    AND TRY_CAST(
      REGEXP_REPLACE(
        REGEXP_REPLACE(TRIM(precio), '[^0-9,.-]', ''),
        ',',
        '.'
      ) AS DECIMAL(18,2)
    ) > 0
),

validaciones AS (
  SELECT
    *,
    -- 1. Validación de campos nulos críticos
    CASE WHEN supermercado IS NULL THEN 1 ELSE 0 END AS null_supermercado,
    CASE WHEN nombre IS NULL THEN 1 ELSE 0 END AS null_nombre,
    CASE WHEN categoria IS NULL THEN 1 ELSE 0 END AS null_categoria,
    CASE WHEN url IS NULL OR url = '' THEN 1 ELSE 0 END AS null_url,
    
    -- 2. Validación de duplicados (mismo producto, supermercado y fecha)
    COUNT(*) OVER (
      PARTITION BY supermercado, nombre, fecha_extraccion
    ) AS duplicados,
    
    -- 3. Detección de outliers en precios (usando percentiles)
    CASE 
      WHEN precio_num > PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY precio_num) OVER (PARTITION BY categoria)
        THEN 'OUTLIER_ALTO'
      WHEN precio_num < PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY precio_num) OVER (PARTITION BY categoria)
        THEN 'OUTLIER_BAJO'
      ELSE 'PRECIO_NORMAL'
    END AS precio_outlier,
    
    -- 4. Validación de URL (formato básico)
    CASE 
      WHEN url NOT LIKE 'http%' THEN 'URL_INVALIDA'
      ELSE 'URL_VALIDA'
    END AS url_valida,
    
    -- 5. Longitud de campos (detectar descripciones sospechosamente cortas)
    LENGTH(nombre) AS longitud_nombre,
    LENGTH(descripcion) AS longitud_descripcion
    
  FROM datos_validos
)


-- Resumen de problemas de calidad encontrados
SELECT
  'Registros con supermercado nulo' AS tipo_validacion,
  SUM(null_supermercado) AS registros_problema,
  CONCAT(ROUND(100.0 * SUM(null_supermercado) / COUNT(*), 2), '%') AS porcentaje
FROM validaciones

UNION ALL

SELECT
  'Registros con nombre nulo',
  SUM(null_nombre),
  CONCAT(ROUND(100.0 * SUM(null_nombre) / COUNT(*), 2), '%')
FROM validaciones

UNION ALL

SELECT
  'Registros con categoría nula',
  SUM(null_categoria),
  CONCAT(ROUND(100.0 * SUM(null_categoria) / COUNT(*), 2), '%')
FROM validaciones

UNION ALL

SELECT
  'Registros con URL inválida',
  SUM(null_url),
  CONCAT(ROUND(100.0 * SUM(null_url) / COUNT(*), 2), '%')
FROM validaciones

UNION ALL

SELECT
  'Registros duplicados',
  SUM(CASE WHEN duplicados > 1 THEN 1 ELSE 0 END),
  CONCAT(ROUND(100.0 * SUM(CASE WHEN duplicados > 1 THEN 1 ELSE 0 END) / COUNT(*), 2), '%')
FROM validaciones

UNION ALL

SELECT
  'Outliers de precio (alto)',
  SUM(CASE WHEN precio_outlier = 'OUTLIER_ALTO' THEN 1 ELSE 0 END),
  CONCAT(ROUND(100.0 * SUM(CASE WHEN precio_outlier = 'OUTLIER_ALTO' THEN 1 ELSE 0 END) / COUNT(*), 2), '%')
FROM validaciones

UNION ALL

SELECT
  'Nombres muy cortos (<3 caracteres)',
  SUM(CASE WHEN longitud_nombre < 3 THEN 1 ELSE 0 END),
  CONCAT(ROUND(100.0 * SUM(CASE WHEN longitud_nombre < 3 THEN 1 ELSE 0 END) / COUNT(*), 2), '%')
FROM validaciones

ORDER BY registros_problema DESC;

In [0]:
-- ========================================
-- MERGE INCREMENTAL DIARIO - CAPA SILVER
-- ========================================
-- Estrategia: 
--   - Deduplica múltiples scrapers del mismo día (mantiene último precio)
--   - Mantiene histórico: 1 fila por producto por fecha
--   - Clave compuesta: (producto_id, fecha_extraccion)

-- Crear schema Silver si no existe
CREATE SCHEMA IF NOT EXISTS supermercadoetl_prod.silver;

-- Crear tabla Silver inicial si no existe
CREATE TABLE IF NOT EXISTS supermercadoetl_prod.silver.precios_productos (
  producto_id STRING COMMENT 'Hash MD5(supermercado||nombre) - identificador único del producto',
  supermercado STRING COMMENT 'Nombre del supermercado',
  categoria STRING COMMENT 'Categoría del producto',
  nombre STRING COMMENT 'Nombre del producto',
  descripcion STRING COMMENT 'Descripción detallada del producto',
  url STRING COMMENT 'URL del producto en el sitio web',
  precio_normalizado DECIMAL(18,2) COMMENT 'Precio normalizado en formato numérico',
  fecha_extraccion DATE COMMENT 'Fecha de extracción del precio',
  fecha_actualizacion TIMESTAMP COMMENT 'Timestamp de última actualización'
)
USING DELTA
COMMENT 'Tabla Silver con precios de productos normalizados - histórico diario';

-- MERGE: Cargar datos desde Bronze
MERGE INTO supermercadoetl_prod.silver.precios_productos AS target
USING (
  SELECT
    -- Surrogate key SIN fecha (permite histórico)
    MD5(CONCAT(supermercado, '|', nombre)) AS producto_id,
    supermercado,
    categoria,
    nombre,
    descripcion,
    url,
    -- Precio normalizado
    CAST(
      REGEXP_REPLACE(
        REGEXP_REPLACE(TRIM(precio), '[^0-9,.-]', ''),
        ',',
        '.'
      ) AS DECIMAL(18,2)
    ) AS precio_normalizado,
    fecha_extraccion,
    CURRENT_TIMESTAMP() AS fecha_actualizacion
  FROM supermercadoetl_prod.bronze.precios_scraping
  WHERE 
    -- Filtros de calidad
    TRY_CAST(
      REGEXP_REPLACE(
        REGEXP_REPLACE(TRIM(precio), '[^0-9,.-]', ''),
        ',',
        '.'
      ) AS DECIMAL(18,2)
    ) IS NOT NULL
    AND TRY_CAST(
      REGEXP_REPLACE(
        REGEXP_REPLACE(TRIM(precio), '[^0-9,.-]', ''),
        ',',
        '.'
      ) AS DECIMAL(18,2)
    ) > 0
    AND supermercado IS NOT NULL
    AND nombre IS NOT NULL
    AND categoria IS NOT NULL
) AS source
ON target.producto_id = source.producto_id 
   AND target.fecha_extraccion = source.fecha_extraccion

-- Si existe (mismo producto, mismo día): actualizar con último precio
WHEN MATCHED THEN UPDATE SET
  target.precio_normalizado = source.precio_normalizado,
  target.descripcion = source.descripcion,
  target.url = source.url,
  target.categoria = source.categoria,
  target.fecha_actualizacion = source.fecha_actualizacion

-- Si no existe (nuevo producto o nueva fecha): insertar
WHEN NOT MATCHED THEN INSERT (
  producto_id,
  supermercado,
  categoria,
  nombre,
  descripcion,
  url,
  precio_normalizado,
  fecha_extraccion,
  fecha_actualizacion
) VALUES (
  source.producto_id,
  source.supermercado,
  source.categoria,
  source.nombre,
  source.descripcion,
  source.url,
  source.precio_normalizado,
  source.fecha_extraccion,
  source.fecha_actualizacion
);

In [0]:
-- NOTA: Ejecutar solo si necesitas recrear la tabla desde cero
-- (por ejemplo, si cambiaste la lógica del producto_id)

-- Eliminar tabla existente
DROP TABLE IF EXISTS supermercadoetl_prod.silver.precios_productos;

-- Ahora ejecuta la celda anterior (MERGE) para recrear la tabla limpia

In [0]:
-- Verificar la creación de la tabla Silver

-- 1. Contar registros
SELECT COUNT(*) AS total_registros
FROM supermercadoetl_prod.silver.precios_productos;

In [0]:
-- Ver esquema de la tabla
DESCRIBE supermercadoetl_prod.silver.precios_productos;

In [0]:
-- Ver muestra de datos
SELECT *
FROM supermercadoetl_prod.silver.precios_productos
ORDER BY precio_normalizado DESC
LIMIT 10;

In [0]:
-- =====================================================
-- CÓMO FUNCIONA EL MERGE EN TU PIPELINE DIARIO
-- =====================================================
--
-- ESCENARIO 1: Primera carga del día (mañana 11-04-2026)
--   Bronze tiene: 12 productos con fecha_extraccion='2026-04-11'
--   MERGE hace: 12 INSERTS (nuevas fechas)
--   Resultado: 24 filas totales (12 del 10-04 + 12 del 11-04)
--
-- ESCENARIO 2: Re-ejecución del pipeline el mismo día
--   Bronze tiene: 12 productos con fecha_extraccion='2026-04-11'
--   MERGE hace: 12 UPDATES (mismo producto_id + fecha_extraccion)
--   Resultado: 24 filas (sin duplicados, precios actualizados)
--
-- ESCENARIO 3: Nuevo producto aparece
--   Bronze tiene: 13 productos (1 nuevo) con fecha='2026-04-12'
--   MERGE hace: 13 INSERTS
--   Resultado: 37 filas (histórico completo)
--
-- ESCENARIO 4: Producto desaparece del scraper
--   Bronze NO tiene el producto X con fecha='2026-04-12'
--   MERGE hace: Nada (no toca filas existentes)
--   Resultado: Producto X sigue en Silver con última fecha conocida
--
-- =====================================================
-- CLAVE COMPUESTA (PK): (producto_id, fecha_extraccion)
-- =====================================================
--   producto_id = MD5(supermercado || nombre)
--   Mismo valor para un producto a través del tiempo
--   
--   fecha_extraccion = Fecha del scraping
--   Permite múltiples filas por producto (histórico)
--
-- EJEMPLO DE HISTÓRICO:
--   producto_id='c46b94f1...' | fecha_extraccion='2026-04-10' | precio=21999.00
--   producto_id='c46b94f1...' | fecha_extraccion='2026-04-11' | precio=22499.00
--   producto_id='c46b94f1...' | fecha_extraccion='2026-04-12' | precio=21799.00
--   ↳ Mismo producto, 3 días diferentes, 3 precios distintos
--
-- =====================================================

-- Verificar productos únicos actuales
SELECT 
  COUNT(DISTINCT producto_id) AS productos_unicos,
  COUNT(DISTINCT fecha_extraccion) AS fechas_distintas,
  COUNT(*) AS total_filas,
  MIN(fecha_extraccion) AS primer_registro,
  MAX(fecha_extraccion) AS ultimo_registro
FROM supermercadoetl_prod.silver.precios_productos;

In [0]:
-- SIMULACIÓN: ¿Qué pasará mañana (11-04-2026) cuando corra el pipeline?
-- 
-- Supongamos que:
--   - Bronze carga los mismos 12 productos
--   - Algunos precios cambiaron
--   - fecha_extraccion = '2026-04-11'

WITH simulacion_manana AS (
  SELECT
    MD5(CONCAT(supermercado, '|', nombre)) AS producto_id,
    supermercado,
    categoria,
    nombre,
    descripcion,
    url,
    -- Simulamos cambios de precio (+5% a todos)
    precio_normalizado * 1.05 AS precio_normalizado,
    DATE '2026-04-11' AS fecha_extraccion,  -- Nueva fecha
    CURRENT_TIMESTAMP() AS fecha_actualizacion
  FROM supermercadoetl_prod.silver.precios_productos
  WHERE fecha_extraccion = DATE '2026-04-10'
)

-- Vista previa: Así se vería el histórico después del MERGE de mañana
SELECT 
  nombre,
  producto_id,
  fecha_extraccion,
  precio_normalizado,
  'REGISTRO ACTUAL' AS tipo
FROM supermercadoetl_prod.silver.precios_productos

UNION ALL

SELECT
  nombre,
  producto_id,
  fecha_extraccion,
  precio_normalizado,
  'NUEVO MAÑANA (simulado)' AS tipo
FROM simulacion_manana

ORDER BY nombre, fecha_extraccion
LIMIT 20;

-- RESULTADO ESPERADO:
--   24 filas: 12 del 10-04 + 12 del 11-04
--   Mismo producto_id, diferentes fechas y precios